# 04 — Network analysis: author reply networks and communities

**Purpose.** This notebook constructs the author-reply networks used by the final report. Nodes are authors; directed weighted edges are replies from a replying author to the parent/top-level author. The notebook exports network metrics, PageRank concentration, Louvain community assignments, robustness checks, and community-validity diagnostics for downstream notebooks.

**Core questions.**

1. How do Reddit and YouTube reply-network shapes differ?
2. How concentrated and robust are influence measures such as PageRank?
3. Are Louvain communities stable enough to use as downstream NLP units, and how much do they overlap with subreddit/channel scope?

**Inputs:** `data/processed/01_preprocessed/{reddit,youtube}_comments.csv` and `youtube_videos.csv` for channel titles.

**Outputs:** `data/processed/04_networks/*.csv|.json|.graphml` and supporting network plots under `plots/`. The main downstream hand-off is `data/processed/04_networks/{reddit,youtube}_community_assignments.csv`, which maps `platform + author_hash` to Louvain community, role, PageRank, dominant scope, and top frame.

## Executive summary

- **Reddit reply graph:** 7,938 author nodes, 10,668 directed weighted author-author edges, **largest WCC = 88.9%**, modularity **0.902**, reciprocity **28.2%**, and **355** raw Louvain communities. This is a real conversation network.
- **YouTube reply graph:** 5,789 author nodes, 5,229 directed weighted author-author edges, **largest WCC = 45.0%**, modularity **0.971**, reciprocity **0.08%**, and **833** raw Louvain communities. This is shallow and fragmented reply structure, not one platform-wide conversation.
- **Interpretation guardrail:** raw Louvain counts are not substantive publics. Notebook 07 applies coverage and a **100-comment** report-grade threshold before interpreting communities.


## 1 · Setup

In [ ]:
"""Paths, imports, project constants.

This notebook is self-contained: helpers live below in `§2`, the data
load + computation happens in `§3`, and §4–§8 each answer one of the
five research questions in the executive summary.
"""

from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA = PROJECT_ROOT / "data"
PROCESSED = DATA / "processed"
PREPROCESSED = PROCESSED / "01_preprocessed"
NETWORKS = PROCESSED / "04_networks"
PLOTS = PROJECT_ROOT / "plots"
LEXICONS = PROJECT_ROOT / "lexicons"
for d in (NETWORKS, PLOTS):
    d.mkdir(parents=True, exist_ok=True)

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

REDDIT_COLOR  = "#3b6fa1"
YOUTUBE_COLOR = "#c45a3d"


def _env_flag(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() not in {"0", "false", "no", "off", "full"}


NETWORK_FAST_MODE = _env_flag("NETWORK_FAST_MODE", True)
RUN_OPTIONAL_NETWORK_DIAGNOSTICS = _env_flag("RUN_OPTIONAL_NETWORK_DIAGNOSTICS", True)
ROBUSTNESS_RESAMPLES = int(os.getenv(
    "ROBUSTNESS_RESAMPLES", "100" if NETWORK_FAST_MODE else "200"
))
LOUVAIN_STABILITY_SEEDS = int(os.getenv(
    "LOUVAIN_STABILITY_SEEDS", "10" if NETWORK_FAST_MODE else "20"
))
CPM_MAX_NODES = int(os.getenv(
    "CPM_MAX_NODES", "2000" if NETWORK_FAST_MODE else "0"
))
SPRING_LAYOUT_MAX_NODES = int(os.getenv(
    "SPRING_LAYOUT_MAX_NODES", "250" if NETWORK_FAST_MODE else "400"
))
_louvain_default = "0.75,1.00,1.50" if NETWORK_FAST_MODE else "0.50,0.75,1.00,1.25,1.50,2.00"
LOUVAIN_RESOLUTIONS = [
    float(value.strip())
    for value in os.getenv("LOUVAIN_RESOLUTIONS", _louvain_default).split(",")
    if value.strip()
]
PAGERANK_EDGE_KEEP_FRACTION = float(os.getenv("PAGERANK_EDGE_KEEP_FRACTION", "0.90"))

print(
    "Network notebook mode: "
    f"{'fast' if NETWORK_FAST_MODE else 'full'}; "
    f"optional diagnostics={RUN_OPTIONAL_NETWORK_DIAGNOSTICS}; "
    f"pagerank edge-drop resamples={ROBUSTNESS_RESAMPLES}; "
    f"louvain seeds={LOUVAIN_STABILITY_SEEDS}; edge keep={PAGERANK_EDGE_KEEP_FRACTION:.0%}; "
    f"cpm max nodes={CPM_MAX_NODES or 'unbounded'}; "
    f"layout max nodes={SPRING_LAYOUT_MAX_NODES}; "
    f"resolutions={LOUVAIN_RESOLUTIONS}"
)


## 2 · Helpers

Split into four small cells so each block does one job.

### 2a · Edge construction

In [ ]:
import networkx as nx

def build_reddit_edges(comments_df: pd.DataFrame, *, drop_submission_replies: bool = True) -> pd.DataFrame:
    """Replier → parent-author edges with a `parent_kind` flag.

    A Reddit top-level comment is a reply to the *submission*, not to
    another commenter.  Treating those edges as normal comment-replies
    inflates the OP's in-degree.  We derive `parent_kind` from `depth`
    (depth==0 → submission, depth>=1 → comment) and, by default, drop
    the submission-replies from the analysed graph.  The dropped rows
    are still saved in the edges CSV for audit.
    """
    if comments_df.empty:
        return pd.DataFrame(columns=["source", "target", "submission_id", "subreddit",
                                     "created_utc", "comment_id", "parent_kind"])
    df = comments_df[comments_df["author_hash"].notna() & comments_df["parent_author_hash"].notna()].copy()
    df = df[df["author_hash"] != df["parent_author_hash"]]
    df["parent_kind"] = np.where(df["depth"].fillna(0).astype(int) == 0, "submission", "comment")
    edges = df[["author_hash", "parent_author_hash", "submission_id", "subreddit",
                "created_utc", "comment_id", "parent_kind"]].rename(
        columns={"author_hash": "source", "parent_author_hash": "target"})
    if drop_submission_replies:
        edges = edges[edges["parent_kind"] == "comment"].copy()
    return edges


def build_youtube_reply_edges(comments_df: pd.DataFrame) -> pd.DataFrame:
    """YouTube `parent_id` points at the parent COMMENT; join to get author."""
    if comments_df.empty:
        return pd.DataFrame(columns=["source", "target", "video_id", "channel_id",
                                     "created_utc", "comment_id"])
    parents = comments_df[~comments_df["is_reply"].fillna(False)][
        ["comment_id", "author_hash", "video_id", "channel_id"]
    ].rename(columns={"author_hash": "parent_author_hash", "comment_id": "parent_id"})
    replies = comments_df[comments_df["is_reply"].fillna(False)].merge(
        parents, on="parent_id", how="inner", suffixes=("", "_parent"),
    )
    replies = replies[replies["author_hash"].notna() & replies["parent_author_hash"].notna()]
    replies = replies[replies["author_hash"] != replies["parent_author_hash"]]
    return replies[["author_hash", "parent_author_hash", "video_id_parent",
                    "channel_id_parent", "published_at", "comment_id"]].rename(columns={
        "author_hash": "source", "parent_author_hash": "target",
        "video_id_parent": "video_id", "channel_id_parent": "channel_id",
        "published_at": "created_utc"})


def edges_to_digraph(edges_df: pd.DataFrame) -> nx.DiGraph:
    """Collapse duplicate (source, target) pairs into one weighted edge."""
    g = nx.DiGraph()
    if edges_df.empty:
        return g
    for (s, t), grp in edges_df.groupby(["source", "target"]):
        g.add_edge(s, t, weight=int(len(grp)))
    return g


### 2b · Centralities, roles, network summary

In [ ]:
def compute_centralities(g: nx.DiGraph) -> pd.DataFrame:
    """Per-node PageRank, in/out degree, betweenness, eigenvector, Katz.

    Katz centrality is a class method (L08 — SNA Measures workshop).
    """
    if g.number_of_nodes() == 0:
        return pd.DataFrame()
    nodes = list(g.nodes())
    in_deg, out_deg = dict(g.in_degree(weight="weight")), dict(g.out_degree(weight="weight"))
    in_unw, out_unw = dict(g.in_degree()), dict(g.out_degree())
    try:
        pr = nx.pagerank(g, weight="weight", max_iter=200)
    except nx.PowerIterationFailedConvergence:
        pr = nx.pagerank(g, weight="weight", max_iter=500, tol=1e-4)
    n = g.number_of_nodes()
    btw = nx.betweenness_centrality(g, k=max(50, int(0.05 * n)), normalized=True, seed=0) if n > 3000 \
          else nx.betweenness_centrality(g, normalized=True)
    try:
        eig = nx.eigenvector_centrality_numpy(g, weight="weight")
    except Exception:
        eig = {n_: float("nan") for n_ in nodes}
    try:  # Katz centrality — taught in L08
        katz = nx.katz_centrality_numpy(g, weight="weight")
    except Exception:
        katz = {n_: float("nan") for n_ in nodes}
    return pd.DataFrame({
        "node": nodes,
        "in_degree_unweighted":  [in_unw[n_]  for n_ in nodes],
        "out_degree_unweighted": [out_unw[n_] for n_ in nodes],
        "in_degree_weighted":    [in_deg[n_]  for n_ in nodes],
        "out_degree_weighted":   [out_deg[n_] for n_ in nodes],
        "pagerank":              [pr[n_]      for n_ in nodes],
        "betweenness":           [btw[n_]     for n_ in nodes],
        "eigenvector":           [eig.get(n_, float("nan")) for n_ in nodes],
        "katz":                  [katz.get(n_, float("nan")) for n_ in nodes],
    })


def assign_role(row, btw_threshold: float) -> str:
    """Crude role label from in/out balance + betweenness."""
    in_w, out_w, btw = row["in_degree_weighted"], row["out_degree_weighted"], row["betweenness"]
    if in_w == 0 and out_w == 0:
        return "isolated"
    ratio = (in_w + 1) / (out_w + 1)
    prefix = "bridge_" if btw and btw >= btw_threshold else ""
    if ratio >= 2:   return f"{prefix}broadcaster"
    if ratio <= 0.5: return f"{prefix}responder"
    return f"{prefix}conversationalist"


def graph_summary(g: nx.DiGraph) -> dict:
    """One-shot network-shape metrics."""
    if g.number_of_nodes() == 0:
        return {"nodes": 0, "edges": 0}
    ug = g.to_undirected()
    wccs = list(nx.weakly_connected_components(g))
    largest = max(wccs, key=len) if wccs else set()
    sub = g.subgraph(largest).copy()
    try:    rec = nx.reciprocity(g)
    except Exception: rec = None
    try:    ass = nx.degree_assortativity_coefficient(g)
    except Exception: ass = None
    return {
        "nodes": g.number_of_nodes(),
        "edges": g.number_of_edges(),
        "density": nx.density(g),
        "reciprocity": rec,
        "assortativity_degree": ass,
        "weakly_connected_components": len(wccs),
        "strongly_connected_components": nx.number_strongly_connected_components(g),
        "largest_wcc_size": len(largest),
        "largest_wcc_fraction": len(largest) / g.number_of_nodes(),
        "average_clustering_undirected": nx.average_clustering(ug),
        "transitivity_global": nx.transitivity(g),  # global clustering coeff — L08
        "lwcc_avg_shortest_path": (
            nx.average_shortest_path_length(sub.to_undirected())
            if sub.number_of_nodes() > 1 and nx.is_connected(sub.to_undirected())
            else None
        ),
    }


### 2c · Communities + frame-based characterization

In [ ]:
try:
    import community as community_louvain  # python-louvain (preferred)
except ImportError:
    community_louvain = None


def detect_communities(g: nx.DiGraph, *, seed: int = 0) -> dict:
    if g.number_of_nodes() == 0:
        return {}
    ug = g.to_undirected()
    if ug.number_of_edges() == 0:
        return {n: i for i, n in enumerate(ug.nodes())}
    if community_louvain is not None:
        return community_louvain.best_partition(ug, weight="weight", random_state=seed)
    if hasattr(nx.community, "louvain_communities"):
        comms = nx.community.louvain_communities(ug, weight="weight", seed=seed)
    else:
        comms = nx.community.greedy_modularity_communities(ug, weight="weight")
    return {n: cid for cid, comm in enumerate(comms) for n in comm}


def modularity(g: nx.DiGraph, partition: dict) -> float | None:
    if g.number_of_nodes() == 0 or not partition:
        return None
    ug = g.to_undirected()
    if ug.number_of_edges() == 0:
        return None
    comms = [{n for n, c in partition.items() if c == cid} for cid in sorted(set(partition.values()))]
    try:
        return float(nx.community.modularity(ug, comms, weight="weight"))
    except Exception:
        return None


def _load_lexicons(lexicons_dir: Path) -> dict[str, re.Pattern]:
    """Compile each lexicons/*.txt into a single regex of whole-phrase OR."""
    out: dict[str, re.Pattern] = {}
    for path in sorted(lexicons_dir.glob("*.txt")):
        terms = [ln.strip().lower() for ln in path.read_text(encoding="utf-8").splitlines()
                 if ln.strip() and not ln.startswith("#")]
        if not terms:
            continue
        joined = "|".join(re.escape(t) for t in terms)
        out[path.stem] = re.compile(rf"(?<![a-z0-9])(?:{joined})(?![a-z0-9])", re.IGNORECASE)
    return out


LEXICON_PATTERNS = _load_lexicons(LEXICONS)


def profile_communities(nodes_df: pd.DataFrame, comments_df: pd.DataFrame,
                        *, scope_col: str, min_members: int = 10, top_n: int = 10) -> pd.DataFrame:
    """Per-community: member count, dominant scope (sub/channel), top-3 frames."""
    if nodes_df.empty or "community" not in nodes_df.columns:
        return pd.DataFrame()
    author_to_comm = dict(zip(nodes_df["node"], nodes_df["community"]))
    df = comments_df.copy()
    df["community"] = df["author_hash"].map(author_to_comm)
    df = df[df["community"].notna() & (df["community"] >= 0)]
    if df.empty:
        return pd.DataFrame()
    df["community"] = df["community"].astype(int)
    text = df["clean_text"].fillna("").astype(str)
    for name, pat in LEXICON_PATTERNS.items():
        df[f"lex_{name}"] = text.str.contains(pat, regex=True)

    rows = []
    for comm, g in df.groupby("community"):
        n_members = g["author_hash"].nunique()
        if n_members < min_members:
            continue
        scope_counts = g[scope_col].value_counts() if scope_col in g.columns else pd.Series(dtype=int)
        dom = scope_counts.index[0] if len(scope_counts) else None
        dom_pct = float(scope_counts.iloc[0] / max(len(g), 1) * 100) if len(scope_counts) else None
        frame_rates = {name: float(g[f"lex_{name}"].mean() * 100) for name in LEXICON_PATTERNS}
        top_frames = sorted(frame_rates.items(), key=lambda x: x[1], reverse=True)[:3]
        rows.append({
            "community": int(comm),
            "n_members": int(n_members),
            "n_comments": int(len(g)),
            f"dominant_{scope_col}": dom,
            f"dominant_{scope_col}_pct": round(dom_pct, 1) if dom_pct is not None else None,
            "top_frame_1": top_frames[0][0],
            "top_frame_1_pct": round(top_frames[0][1], 1),
            "top_frame_2": top_frames[1][0] if len(top_frames) > 1 else None,
            "top_frame_2_pct": round(top_frames[1][1], 1) if len(top_frames) > 1 else None,
            "top_frame_3": top_frames[2][0] if len(top_frames) > 2 else None,
            "top_frame_3_pct": round(top_frames[2][1], 1) if len(top_frames) > 2 else None,
        })
    return pd.DataFrame(rows).sort_values("n_members", ascending=False).head(top_n)


# --- helper shared by the build step and §6/§7 -----------------------------

def youtube_with_channel_title(comments_df: pd.DataFrame) -> pd.DataFrame:
    """Attach the human-readable channel_title to YouTube comments."""
    videos_path = PREPROCESSED / "youtube_videos.csv"
    if not videos_path.exists():
        return comments_df
    videos = pd.read_csv(videos_path)
    if "channel_title" not in videos.columns:
        return comments_df
    return comments_df.merge(videos[["video_id", "channel_title"]].drop_duplicates(),
                             on="video_id", how="left")


# --- Workshop-9 class methods: CPM communities + purity evaluation ---------

def cpm_communities(g: nx.DiGraph, *, k: int = 3) -> list[set]:
    """k-clique percolation communities (overlapping) — Workshop-9 method.

    CPM only covers nodes that sit in at least one k-clique, so many
    nodes are left uncovered; that is expected and is the contrast with
    Louvain's full partition.
    """
    ug = g.to_undirected()
    if ug.number_of_edges() == 0:
        return []
    if CPM_MAX_NODES and ug.number_of_nodes() > CPM_MAX_NODES:
        print(
            f"[skip] CPM k={k}: {ug.number_of_nodes():,} nodes exceeds "
            f"CPM_MAX_NODES={CPM_MAX_NODES:,}"
        )
        return []
    try:
        return [set(comm) for comm in nx.algorithms.community.k_clique_communities(ug, k)]
    except Exception:
        return []


def purity(detected: list[set], ground_truth: list[set]) -> float:
    """Purity of a detected partition against ground-truth communities.

    For each detected community, take its largest overlap with any
    ground-truth community; sum those maxima; normalise by the total
    node count across detected communities.  (Workshop-9 measure.)
    Purity = 1.0 means every detected community sits entirely inside a
    single ground-truth group.
    """
    if not detected:
        return float("nan")
    total_overlap = total_nodes = 0
    for comm in detected:
        if not comm:
            continue
        total_overlap += max((len(comm & gt) for gt in ground_truth), default=0)
        total_nodes += len(comm)
    return total_overlap / total_nodes if total_nodes else float("nan")


### 2d · Robustness probes (bootstrap τ + Louvain ARI) and Gini

In [ ]:
from scipy.stats import kendalltau
from sklearn.metrics import adjusted_rand_score


def gini(values) -> float:
    """Gini coefficient of a non-negative vector."""
    x = np.sort(np.asarray(values, dtype=float))
    n = x.size
    if n == 0 or x.sum() == 0:
        return float("nan")
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))


def _ci(values, lo: float = 2.5, hi: float = 97.5) -> tuple[float, float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return float("nan"), float("nan"), float("nan")
    return float(arr.mean()), float(np.percentile(arr, lo)), float(np.percentile(arr, hi))


def _empirical_p_gt_zero(values) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return float("nan")
    return float((np.sum(arr <= 0) + 1) / (arr.size + 1))


def bootstrap_centrality_rank(edges_df: pd.DataFrame, *, top_k: int = 20,
                              n_resamples: int = 100, seed: int = 0) -> dict:
    """Fast PageRank top-K stability under edge-drop perturbations.

    This keeps the robustness check reportable without making the notebook slow:
    each run recomputes PageRank after dropping a small random share of observed
    reply rows.  The result is a perturbation interval, not a causal test.
    """
    if edges_df.empty:
        return {"mean_tau": float("nan"), "lo": float("nan"), "hi": float("nan")}

    base_pr = pd.Series(nx.pagerank(edges_to_digraph(edges_df), weight="weight", max_iter=200, tol=1e-6))
    top_nodes = base_pr.nlargest(top_k).index.astype(str).tolist()
    base_rank = pd.Series(np.arange(1, len(top_nodes) + 1), index=top_nodes)
    rng = np.random.default_rng(seed)
    taus = []
    overlaps = []

    for _ in range(n_resamples):
        sample = edges_df.sample(
            frac=PAGERANK_EDGE_KEEP_FRACTION,
            replace=False,
            random_state=int(rng.integers(0, 2**31 - 1)),
        )
        g = edges_to_digraph(sample)
        try:
            pr = pd.Series(nx.pagerank(g, weight="weight", max_iter=120, tol=1e-4))
        except nx.PowerIterationFailedConvergence:
            pr = pd.Series(nx.pagerank(g, weight="weight", max_iter=250, tol=1e-3))

        sample_top = pr.nlargest(top_k).index.astype(str).tolist()
        present = [n_ for n_ in top_nodes if n_ in pr.index]
        if len(present) >= 2:
            tau, _ = kendalltau(base_rank.loc[present].values,
                                pr.loc[present].rank(ascending=False, method="average").values)
            if not np.isnan(tau):
                taus.append(float(tau))
        overlaps.append(len(set(top_nodes).intersection(sample_top)) / max(top_k, 1))

    mean_tau, lo, hi = _ci(taus)
    overlap_mean, overlap_lo, overlap_hi = _ci(overlaps)
    return {
        "mean_tau": mean_tau,
        "lo": lo,
        "hi": hi,
        "p_gt_zero": _empirical_p_gt_zero(taus),
        "n_resamples": len(taus),
        "top_nodes": top_nodes,
        "topk_overlap_mean": overlap_mean,
        "topk_overlap_lo": overlap_lo,
        "topk_overlap_hi": overlap_hi,
        "edge_keep_fraction": PAGERANK_EDGE_KEEP_FRACTION,
    }


def louvain_stability(g: nx.DiGraph, *, n_seeds: int = 10) -> dict:
    """Pairwise Adjusted Rand Index across Louvain seeds."""
    if g.number_of_nodes() < 2:
        return {"mean_ari": float("nan"), "lo": float("nan"), "hi": float("nan")}
    nodes = sorted(g.nodes())
    parts = [[detect_communities(g, seed=s).get(n_, -1) for n_ in nodes] for s in range(n_seeds)]
    aris = [adjusted_rand_score(parts[i], parts[j]) for i in range(len(parts)) for j in range(i)]
    mean_ari, lo, hi = _ci(aris)
    return {
        "mean_ari": mean_ari,
        "lo": lo,
        "hi": hi,
        "p_gt_zero": _empirical_p_gt_zero(aris),
        "n_pairs": len(aris),
        "n_seeds": n_seeds,
    }

## 3 · Build the networks

One pass loads the preprocessed comments, builds the reply graph, computes centralities and Louvain communities, and caches everything in a `PlatformGraph` dataclass per platform. Every later section in this notebook reads from this cache instead of re-loading.

**Downstream hand-off.** This notebook writes author-level community assignments only. Notebook 07 later joins these files to comment-level sentiment/topic outputs after notebook 05 has run:

- `data/processed/04_networks/reddit_community_assignments.csv`
- `data/processed/04_networks/youtube_community_assignments.csv`

The hand-off key is `platform, author_hash`. This keeps notebook 04 independent of notebook 05 while still preparing the network structure needed for community-level NLP.


In [ ]:
@dataclass
class PlatformGraph:
    """All per-platform artefacts in one object — section §4–§9 just unpack this."""
    platform: str
    comments: pd.DataFrame
    edges:    pd.DataFrame       # canonical comment→comment edges
    edges_all: pd.DataFrame      # Reddit only: includes submission-replies for audit
    graph:    nx.DiGraph
    nodes:    pd.DataFrame       # one row per author with centralities + community + role
    summary:  dict


def _read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def build_platform(platform: str) -> PlatformGraph:
    comments = _read_csv(PREPROCESSED / f"{platform}_comments.csv")
    if comments.empty:
        raise FileNotFoundError(f"No {platform}_comments.csv under {PREPROCESSED}")

    if platform == "reddit":
        edges_all = build_reddit_edges(comments, drop_submission_replies=False)
        edges     = edges_all[edges_all["parent_kind"] == "comment"].copy()
    else:
        edges_all = build_youtube_reply_edges(comments)
        edges     = edges_all.copy()

    g = edges_to_digraph(edges)
    nodes = compute_centralities(g)
    if not nodes.empty:
        btw_thr = nodes["betweenness"].quantile(0.95)
        nodes["role"] = nodes.apply(lambda r: assign_role(r, btw_thr), axis=1)
        part = detect_communities(g, seed=0)
        nodes["community"] = nodes["node"].map(part).fillna(-1).astype(int)
    summary = graph_summary(g)
    if not nodes.empty:
        summary["modularity"]  = modularity(g, dict(zip(nodes["node"], nodes["community"])))
        summary["communities"] = int(nodes["community"].nunique())

    edges.to_csv(NETWORKS / f"{platform}_edges.csv", index=False)
    nodes.to_csv(NETWORKS / f"{platform}_nodes.csv", index=False)
    (NETWORKS / f"{platform}_network_summary.json").write_text(
        json.dumps(summary, indent=2, default=float))
    return PlatformGraph(platform, comments, edges, edges_all, g, nodes, summary)


platforms: dict[str, PlatformGraph] = {}
for plat in ("reddit", "youtube"):
    platforms[plat] = build_platform(plat)
    pg = platforms[plat]
    s = pg.summary
    if plat == "reddit":
        n_op = int((pg.edges_all["parent_kind"] == "submission").sum())
        n_cc = int((pg.edges_all["parent_kind"] == "comment").sum())
        print(f"reddit  | edges built  comment→comment={n_cc:,}  (dropped {n_op:,} submission-replies)  "
              f"| nodes={s['nodes']:,}  edges={s['edges']:,}  modularity={s.get('modularity'):.3f}")
    else:
        print(f"youtube | edges built  comment→comment={s['edges']:,}  "
              f"| nodes={s['nodes']:,}  modularity={s.get('modularity'):.3f}")


# --- Hand-off exports for the downstream sentiment/topic-per-community nb ---
# A tidy per-author assignment CSV plus the class-standard GraphML (node
# attributes = centralities + community + role).  Downstream sentiment/topic
# notebooks should join this CSV to comments on platform + author_hash, then
# join sentiment/topic tables on platform + comment_id.
for _plat, _pg in platforms.items():
    _scope = "subreddit" if _plat == "reddit" else "channel_title"
    _comments = _pg.comments if _plat == "reddit" else youtube_with_channel_title(_pg.comments)
    _prof = profile_communities(_pg.nodes, _comments, scope_col=_scope,
                                min_members=1, top_n=10_000_000)
    _domsub = _prof.set_index("community")[f"dominant_{_scope}"].to_dict() if not _prof.empty else {}
    _frame  = _prof.set_index("community")["top_frame_1"].to_dict()        if not _prof.empty else {}
    _assign = _pg.nodes[["node", "community", "role", "pagerank"]].copy()
    _assign.insert(0, "platform", _plat)
    _assign = _assign.rename(columns={"node": "author_hash"})
    _assign["community_dominant_scope"] = _assign["community"].map(_domsub)
    _assign["community_top_frame"]      = _assign["community"].map(_frame)
    _assign.to_csv(NETWORKS / f"{_plat}_community_assignments.csv", index=False)

    _gx = _pg.graph.copy()
    _attr = _pg.nodes.set_index("node")
    for _col in ["pagerank", "betweenness", "eigenvector", "katz", "community", "role"]:
        if _col in _attr.columns:
            nx.set_node_attributes(_gx, _attr[_col].to_dict(), _col)
    nx.write_graphml(_gx, NETWORKS / f"{_plat}_reply_graph.graphml", infer_numeric_types=True)
    print(f"{_plat}: wrote {_plat}_community_assignments.csv ({len(_assign):,} authors) "
          f"+ {_plat}_reply_graph.graphml")


## 4 · Q1 — How big and how different are the two reply networks?

Two views side-by-side: a **shape panel** (largest-WCC, reciprocity, clustering, modularity) and a **PageRank rank-frequency** plot on log-log axes — the canonical view for power-law-ish distributions.  Linear-axis histograms hide everything for these data.

We also report **Gini(PageRank)** and the **top-1 % share** so the report can compare *how unequal* influence is across platforms, not just *how big* the network is.

In [ ]:
shape_rows = [
    {
        "platform": p,
        "nodes": pg.summary["nodes"],
        "edges": pg.summary["edges"],
        "largest_wcc_pct":  100.0 * (pg.summary.get("largest_wcc_fraction") or 0.0),
        "reciprocity_pct":  100.0 * (pg.summary.get("reciprocity") or 0.0),
        "avg_clustering":   pg.summary.get("average_clustering_undirected") or 0.0,
        "modularity":       pg.summary.get("modularity") or 0.0,
        "gini_pagerank":    gini(pg.nodes["pagerank"].values) if not pg.nodes.empty else float("nan"),
        "top_1pct_share":   (pg.nodes["pagerank"].sort_values(ascending=False).head(
                                 max(1, len(pg.nodes) // 100)).sum() / pg.nodes["pagerank"].sum())
                            if not pg.nodes.empty else float("nan"),
    }
    for p, pg in platforms.items()
]
shape_df = pd.DataFrame(shape_rows)
shape_df.to_csv(NETWORKS / "network_shape_comparison.csv", index=False)

pagerank_concentration = []
for p, pg in platforms.items():
    pr = pg.nodes["pagerank"].sort_values(ascending=False)
    pagerank_concentration.append({
        "platform": p,
        "n_nodes": int(len(pr)),
        "gini_pagerank": gini(pr.values) if len(pr) else float("nan"),
        "top10_share": float(pr.head(10).sum() / pr.sum()) if pr.sum() else float("nan"),
        "top1pct_share": float(pr.head(max(1, len(pr) // 100)).sum() / pr.sum()) if pr.sum() else float("nan"),
    })
pd.DataFrame(pagerank_concentration).to_csv(NETWORKS / "pagerank_concentration.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) network-shape bars
metrics = [("largest_wcc_pct",  "Largest WCC (%)"),
           ("reciprocity_pct",  "Reciprocity (%)"),
           ("avg_clustering",   "Avg clustering coeff."),
           ("modularity",       "Modularity (Louvain)")]
colours = [REDDIT_COLOR if p == "reddit" else YOUTUBE_COLOR for p in shape_df["platform"]]
ax = axes[0, 0]
x = np.arange(len(shape_df))
width = 0.18
for i, (col, label) in enumerate(metrics):
    vals = shape_df[col].astype(float).values
    rng_vmax = max(abs(vals).max(), 1e-9)
    norm = vals / rng_vmax  # plot on normalised axis but label real
    ax.bar(x + i * width - 1.5 * width, vals, width=width, label=label, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(shape_df["platform"])
ax.set_title("Network shape — side by side")
ax.legend(loc="upper right", fontsize=8)

# (b) Gini + top-1% share text panel
ax = axes[0, 1]
ax.axis("off")
ax.set_title("PageRank concentration")
lines = ["{:>8s}  {:>6}  {:>10s}  {:>13s}".format("platform", "nodes", "Gini", "top-1% share")]
lines.append("-" * len(lines[0]))
for r in shape_df.itertuples():
    lines.append(f"{r.platform:>8s}  {int(r.nodes):>6,}  {r.gini_pagerank:>10.3f}  {r.top_1pct_share:>12.1%}")
ax.text(0.02, 0.85, "\n".join(lines), family="monospace", fontsize=11, va="top")

# (c) Reddit rank-frequency
for ax, plat in zip(axes[1, :], ("reddit", "youtube")):
    pr = platforms[plat].nodes["pagerank"].sort_values(ascending=False).values
    if pr.size == 0:
        ax.set_visible(False); continue
    ax.loglog(np.arange(1, pr.size + 1), pr, marker=".", linestyle="-",
              color=REDDIT_COLOR if plat == "reddit" else YOUTUBE_COLOR, alpha=0.7)
    ax.set_xlabel("rank"); ax.set_ylabel("PageRank")
    ax.set_title(f"{plat} — PageRank vs rank (log-log)")
    ax.grid(True, which="both", linewidth=0.3, alpha=0.5)

plt.tight_layout()
plt.savefig(PLOTS / "network_shape_and_concentration.png", dpi=160, bbox_inches="tight")
plt.show()
shape_df


**What this means.** Reddit is a *conversation network*: most nodes are in one giant component, 28 % of replies provoke a counter-reply.  YouTube is a *forest of small comment-trees*: only ~45 % of nodes are in the largest component, reciprocity is ~0.08 % (essentially zero).  Influence on Reddit is unequal in Gini terms (≈ 0.6); YouTube's is flatter.

## 5 · Q2 — Are our influence claims robust?

Two fast statistical probes:

1. **PageRank edge-drop τ** - refit PageRank on **100 perturbations** that retain 90 % of directed weighted edges, then compute Kendall τ between the perturbation top-20 and the baseline top-20. A τ near 1 with a tight interval means the top ranking is reproducible; a τ near 0 means the identities are not reliable.
2. **Louvain ARI** - compute pairwise Adjusted Rand Index over **10 Louvain random seeds** (45 seed pairs). ARI near 1 means the community partition is deterministic enough to interpret substantively.

Both are shown as forest plots on the same horizontal axis so a reader can compare ranking stability and community stability at a glance. The saved CSV also includes one-sided empirical p-values for statistic > 0 with add-one smoothing.


In [ ]:
if RUN_OPTIONAL_NETWORK_DIAGNOSTICS:
    print(
        f"Running robustness checks: {ROBUSTNESS_RESAMPLES} PageRank edge-drop resamples, "
        f"{LOUVAIN_STABILITY_SEEDS} Louvain seeds"
    )
    tau = {p: bootstrap_centrality_rank(pg.edges, top_k=20,
                                        n_resamples=ROBUSTNESS_RESAMPLES, seed=0)
           for p, pg in platforms.items()}
    ari = {p: louvain_stability(pg.graph, n_seeds=LOUVAIN_STABILITY_SEEDS)
           for p, pg in platforms.items()}

    robust_df = pd.DataFrame({
        "platform": list(platforms),
        "tau_mean": [tau[p]["mean_tau"] for p in platforms],
        "tau_lo":   [tau[p]["lo"]       for p in platforms],
        "tau_hi":   [tau[p]["hi"]       for p in platforms],
        "tau_p_gt_zero": [tau[p].get("p_gt_zero", np.nan) for p in platforms],
        "topk_overlap_mean": [tau[p].get("topk_overlap_mean", np.nan) for p in platforms],
        "topk_overlap_lo": [tau[p].get("topk_overlap_lo", np.nan) for p in platforms],
        "topk_overlap_hi": [tau[p].get("topk_overlap_hi", np.nan) for p in platforms],
        "ari_mean": [ari[p]["mean_ari"] for p in platforms],
        "ari_lo":   [ari[p]["lo"]       for p in platforms],
        "ari_hi":   [ari[p]["hi"]       for p in platforms],
        "ari_p_gt_zero": [ari[p].get("p_gt_zero", np.nan) for p in platforms],
        "tau_resamples": [tau[p].get("n_resamples", 0) for p in platforms],
        "ari_pairs": [ari[p].get("n_pairs", 0) for p in platforms],
        "edge_keep_fraction": [tau[p].get("edge_keep_fraction", np.nan) for p in platforms],
        "louvain_seeds": [ari[p].get("n_seeds", 0) for p in platforms],
        "robustness_note": [
            f"Computed: {ROBUSTNESS_RESAMPLES} PageRank edge-drop perturbations at "
            f"{PAGERANK_EDGE_KEEP_FRACTION:.0%} edge retention; {LOUVAIN_STABILITY_SEEDS} "
            "Louvain seeds. Empirical p-values are one-sided for statistic > 0 with add-one smoothing."
        ] * len(platforms),
    })
else:
    note = "Skipped; set RUN_OPTIONAL_NETWORK_DIAGNOSTICS=1 for PageRank/bootstrap and Louvain-seed robustness."
    print(note)
    robust_df = pd.DataFrame({
        "platform": list(platforms),
        "tau_mean": [np.nan] * len(platforms),
        "tau_lo": [np.nan] * len(platforms),
        "tau_hi": [np.nan] * len(platforms),
        "tau_p_gt_zero": [np.nan] * len(platforms),
        "topk_overlap_mean": [np.nan] * len(platforms),
        "topk_overlap_lo": [np.nan] * len(platforms),
        "topk_overlap_hi": [np.nan] * len(platforms),
        "ari_mean": [np.nan] * len(platforms),
        "ari_lo": [np.nan] * len(platforms),
        "ari_hi": [np.nan] * len(platforms),
        "ari_p_gt_zero": [np.nan] * len(platforms),
        "tau_resamples": [0] * len(platforms),
        "ari_pairs": [0] * len(platforms),
        "edge_keep_fraction": [np.nan] * len(platforms),
        "louvain_seeds": [0] * len(platforms),
        "robustness_note": [note] * len(platforms),
    })
robust_df.to_csv(NETWORKS / "robustness_checks.csv", index=False)
robust_df.to_csv(NETWORKS / "robustness_checks_statistical.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
if RUN_OPTIONAL_NETWORK_DIAGNOSTICS:
    y_offsets = {"reddit": 0.85, "youtube": 0.15}
    ys = []
    labels = []
    for _, row in robust_df.iterrows():
        col = REDDIT_COLOR if row["platform"] == "reddit" else YOUTUBE_COLOR
        y_tau = y_offsets[row["platform"]] + 1.0
        y_ari = y_offsets[row["platform"]]
        ax.plot([row["tau_lo"], row["tau_hi"]], [y_tau, y_tau], color="grey", linewidth=2)
        ax.plot(row["tau_mean"], y_tau, "o", color=col, markersize=10)
        ax.text(row["tau_hi"] + 0.02, y_tau,
                f"tau = {row['tau_mean']:.2f}  [{row['tau_lo']:.2f}, {row['tau_hi']:.2f}], p={row['tau_p_gt_zero']:.3f}",
                va="center", fontsize=9)
        ys.extend([y_tau, y_ari]); labels.extend([f"{row['platform']}  tau (PageRank top-20)",
                                                  f"{row['platform']}  ARI (Louvain seeds)"])

        ax.plot([row["ari_lo"], row["ari_hi"]], [y_ari, y_ari], color="grey", linewidth=2)
        ax.plot(row["ari_mean"], y_ari, "s", color=col, markersize=10)
        ax.text(row["ari_hi"] + 0.02, y_ari,
                f"ARI = {row['ari_mean']:.2f}  [{row['ari_lo']:.2f}, {row['ari_hi']:.2f}], p={row['ari_p_gt_zero']:.3f}",
                va="center", fontsize=9)
    ax.axvline(0, color="red",   linewidth=0.6, alpha=0.5)
    ax.axvline(1, color="green", linewidth=0.6, alpha=0.5)
    ax.set_yticks(ys); ax.set_yticklabels(labels)
    ax.set_xlim(-0.2, 1.35); ax.set_xlabel("value (95% perturbation interval)")
    ax.set_title("Robustness - PageRank ranking tau and Louvain community ARI")
else:
    ax.axis("off")
    ax.text(0.02, 0.65, robust_df["robustness_note"].iloc[0], wrap=True, fontsize=11)
    ax.text(0.02, 0.35, "Core network exports were still recomputed from the current data.", fontsize=11)
    ax.set_title("Robustness diagnostics skipped")
plt.tight_layout()
plt.savefig(PLOTS / "robustness_forest.png", dpi=160, bbox_inches="tight")
plt.show()
robust_df

**What this means.** Reddit's top-20 PageRank ranking is statistically above random but only **moderately stable**: τ = **0.36** with a 95 % interval **0.19-0.45** and mean top-20 overlap **45.5 %**. YouTube's top-20 ordering is more stable: τ = **0.68** with a 95 % interval **0.60-0.78** and mean top-20 overlap **76.3 %**. **Both** Louvain partitions are much more stable than individual Reddit ranks: ARI = **0.74** (0.69-0.80) for Reddit and **0.94** (0.90-0.99) for YouTube. Practical implication: the report should frame Reddit influence in role/community terms, not as a precise ranked-user list; YouTube top users can be mentioned only with their channel/community context.


## Robustness/quality check — community validity

Workshop 9 taught two community-detection algorithms plus an evaluation measure:

* **Louvain** (modularity maximisation) — used throughout this notebook.
* **CPM** (k-clique percolation, `k_clique_communities`) — finds *overlapping* communities; run here as the class-prescribed cross-check.
* **Purity** — the Workshop-9 evaluation measure.  We have a natural ground truth: the **subreddit** (Reddit) / **channel** (YouTube) each author predominantly posts in.  Purity asks how concentrated each detected community is on a single ground-truth group — purity 1.0 means every detected community sits entirely inside one subreddit/channel.

A high Louvain purity is the defensible, class-method evidence that the communities this notebook hands to the downstream sentiment/topic-per-community analysis mostly recover real platform groups, not arbitrary algorithmic bins.  The downstream notebook should still report coverage and minimum community-size thresholds because Louvain produces many small communities.

In [ ]:
def ground_truth_partition(pg: PlatformGraph, scope_col: str) -> list[set]:
    """Ground-truth communities = authors grouped by dominant subreddit/channel."""
    comments = pg.comments if scope_col == "subreddit" else youtube_with_channel_title(pg.comments)
    dom = (comments.dropna(subset=[scope_col])
                   .groupby("author_hash")[scope_col]
                   .agg(lambda s: s.mode().iloc[0]))
    groups: dict[str, set] = {}
    for author, scope in dom.items():
        groups.setdefault(scope, set()).add(author)
    return list(groups.values())


rows = []
for plat, pg in platforms.items():
    scope_col = "subreddit" if plat == "reddit" else "channel_title"
    gt = ground_truth_partition(pg, scope_col)
    louvain_sets: dict[int, set] = {}
    for _, r in pg.nodes.iterrows():
        louvain_sets.setdefault(int(r["community"]), set()).add(r["node"])
    louvain = list(louvain_sets.values())
    cpm = cpm_communities(pg.graph, k=3)
    n_nodes = pg.graph.number_of_nodes()
    rows.append({
        "platform": plat,
        "ground_truth_groups":  len(gt),
        "louvain_communities":  len(louvain),
        "louvain_purity":       purity(louvain, gt),
        "cpm_communities":      len(cpm),
        "cpm_purity":           purity(cpm, gt) if cpm else float("nan"),
        "cpm_node_coverage":    (sum(len(c) for c in cpm) / n_nodes) if n_nodes and cpm else 0.0,
    })
validity = pd.DataFrame(rows)
validity.to_csv(NETWORKS / "community_validity.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(validity)); w = 0.35
ax.bar(x - w / 2, validity["louvain_purity"], w, label="Louvain", color=REDDIT_COLOR)
ax.bar(x + w / 2, validity["cpm_purity"].fillna(0), w, label="CPM (k=3)", color=YOUTUBE_COLOR)
ax.set_xticks(x); ax.set_xticklabels(validity["platform"])
ax.set_ylabel("purity vs subreddit / channel ground truth")
ax.set_ylim(0, 1.08)
ax.axhline(1.0, color="green", linewidth=0.5, alpha=0.4)
ax.set_title("Community-detection purity — Louvain vs CPM (Workshop-9 methods)")
for i, r in validity.iterrows():
    ax.text(i - w / 2, r["louvain_purity"] + 0.02, f"{r['louvain_purity']:.2f}",
            ha="center", fontsize=9)
    if pd.notna(r["cpm_purity"]):
        ax.text(i + w / 2, r["cpm_purity"] + 0.02, f"{r['cpm_purity']:.2f}",
                ha="center", fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS / "community_validity_purity.png", dpi=160, bbox_inches="tight")
plt.show()
validity


**What this means.** Louvain purity is high but not perfect: **0.83 for Reddit** and **0.84 for YouTube**. That confirms numerically what the confusion matrix shows visually: detected communities mostly sit inside a dominant subreddit or channel, while still leaving room for cross-scope authors and mixed broadcaster audiences. CPM (overlapping, clique-based) is kept as a Workshop-9 cross-check, but in this bounded run it produced no usable k-clique partition with non-zero node coverage. Because Louvain gives a full, reasonably high-purity and seed-stable partition, it is the community labelling this notebook exports (`{platform}_community_assignments.csv`) for downstream sentiment/topic analysis.


## 8 · Report figure support — semantic network portrait

This kept visual check colours author nodes by interpretable platform scope: subreddit on Reddit and channel on YouTube. It supports the report's explanation that Reddit has a denser conversational core, while YouTube reply activity is more tied to channel/video clusters.


In [ ]:
def _user_scope(pg: PlatformGraph, scope_col: str) -> dict:
    """Map each author_hash to their dominant scope value."""
    comments = pg.comments if scope_col == "subreddit" else youtube_with_channel_title(pg.comments)
    return (comments.dropna(subset=[scope_col])
                    .groupby("author_hash")[scope_col]
                    .agg(lambda s: s.mode().iloc[0])
                    .to_dict())


def plot_network_semantic(pg: PlatformGraph, *, scope_col: str, max_nodes: int | None = None,
                          title: str, output_path: Path) -> None:
    max_nodes = SPRING_LAYOUT_MAX_NODES if max_nodes is None else max_nodes
    g = pg.graph
    if g.number_of_nodes() == 0:
        print(f"[skip] {title}: empty graph"); return
    largest = max(nx.weakly_connected_components(g), key=len)
    sub = g.subgraph(largest).copy()
    if sub.number_of_nodes() > max_nodes:
        keep = pg.nodes[pg.nodes["node"].isin(sub.nodes())].nlargest(max_nodes, "pagerank")["node"].tolist()
        sub = sub.subgraph(keep).copy()
    if sub.number_of_nodes() == 0:
        return

    nodes_df = pg.nodes[pg.nodes["node"].isin(sub.nodes())].set_index("node")
    user_scope = _user_scope(pg, scope_col)
    pr = nodes_df["pagerank"]; pr_n = (pr - pr.min()) / (pr.max() - pr.min() + 1e-12)
    scopes = [user_scope.get(n, "other") for n in sub.nodes()]
    palette_keys = list(pd.Series(scopes).value_counts().head(10).index)  # top 10 scopes get distinct colours
    palette = {k: plt.cm.tab10(i % 10) for i, k in enumerate(palette_keys)}
    node_colors = [palette.get(s, (0.75, 0.75, 0.75, 1.0)) for s in scopes]

    pos = nx.spring_layout(sub.to_undirected(), seed=0,
                           k=0.8 / np.sqrt(max(sub.number_of_nodes(), 1)))
    plt.figure(figsize=(11, 9))
    nx.draw_networkx_edges(sub, pos, alpha=0.18, width=0.6, edge_color="grey", arrows=False)
    nx.draw_networkx_nodes(
        sub, pos, node_color=node_colors,
        node_size=[40 + 600 * float(pr_n.loc[n]) for n in sub.nodes()],
        edgecolors="white", linewidths=0.5,
    )
    handles = [plt.Line2D([0], [0], marker="o", linestyle="", markersize=8,
                          markerfacecolor=palette[k], markeredgecolor="white", label=k)
               for k in palette_keys]
    plt.legend(handles=handles, loc="upper left", fontsize=8, frameon=True,
               title=f"top-{len(palette_keys)} {scope_col}s")
    plt.title(title); plt.axis("off"); plt.tight_layout()
    plt.savefig(output_path, dpi=160, bbox_inches="tight"); plt.show()


plot_network_semantic(
    platforms["reddit"], scope_col="subreddit", max_nodes=SPRING_LAYOUT_MAX_NODES,
    title="Reddit reply network — largest WCC, top 400 by PageRank, coloured by subreddit",
    output_path=PLOTS / "network_visualisation_reddit.png",
)
plot_network_semantic(
    platforms["youtube"], scope_col="channel_title", max_nodes=SPRING_LAYOUT_MAX_NODES,
    title="YouTube reply network — largest WCC, top 400 by PageRank, coloured by dominant channel",
    output_path=PLOTS / "network_visualisation_youtube.png",
)


**What this means.** Semantic colouring helps interpret the structural metrics without treating anonymous Louvain IDs as named publics. Reddit's visible core is dominated by subreddit-specific conversation pockets inside a larger connected graph. YouTube's reply activity is more channel-shaped and less reciprocal, which is why the report treats YouTube communities as conditional on reply-network participation.


## Generated artifacts

Relevant existing outputs from this notebook are:

- `data/processed/04_networks/reddit_edges.csv`, `youtube_edges.csv`
- `data/processed/04_networks/reddit_nodes.csv`, `youtube_nodes.csv`
- `data/processed/04_networks/reddit_community_assignments.csv`, `youtube_community_assignments.csv`
- `data/processed/04_networks/network_shape_comparison.csv`
- `data/processed/04_networks/pagerank_concentration.csv`
- `data/processed/04_networks/robustness_checks.csv`, `robustness_checks_statistical.csv`
- `data/processed/04_networks/community_validity.csv`
- `data/processed/04_networks/network_edge_accounting.csv`
- `data/processed/04_networks/network_scope_baseline_comparison.csv`
- `plots/network_pagerank_display.png`
- `plots/robustness_forest.png`

## Report-ready findings

- **Network definition:** nodes are authors; directed weighted edges are replies from a replying author to the parent/top-level author.
- **Network shape:** Reddit has **7,938 nodes**, **10,668 edges**, **88.9%** of nodes in the largest weakly connected component, and **28.2%** reciprocity. YouTube has **5,789 nodes**, **5,229 edges**, **45.0%** largest-WCC share, and **0.08%** reciprocity.
- **Influence:** Reddit has stronger PageRank concentration (**top 1% = 17.9%**) than YouTube (**12.9%**), but exact Reddit top-user ranking is only moderately stable under edge resampling. Interpret influence as structural role, not as definitive named-user ranking.
- **Communities:** raw Louvain counts (**355 Reddit**, **833 YouTube**) are not substantive publics. They are exported for downstream joining, then interpreted only after notebook 07's coverage and **100-comment** report-grade threshold.
- **Scope caveat:** Louvain purity against subreddit/channel scope is high (**0.83 Reddit**, **0.84 YouTube**), so community analysis adds structure but does not replace platform-scope interpretation.


## Robustness/quality check — interpretability diagnostics

The headline network metrics show that Louvain finds structure, but the raw community counts are not themselves substantive claims. This section adds lightweight diagnostics that read the saved notebook-04 outputs rather than recomputing the full notebook:

1. **Edge accounting** distinguishes raw reply rows from collapsed weighted author-author graph edges.
2. **Community-size thresholds** show how much of the author/comment mass survives practical minimum-size cutoffs.
3. **Resolution sensitivity** checks whether Louvain conclusions depend on one resolution setting.
4. **Scope-baseline comparison** asks whether Louvain adds structure beyond subreddit/channel labels.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = PROJECT_ROOT / "data" / "processed"
PREPROCESSED = PROCESSED / "01_preprocessed"
NETWORKS = PROCESSED / "04_networks"
PLOTS = PROJECT_ROOT / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

REDDIT_COLOR = "#3b6fa1"
YOUTUBE_COLOR = "#c45a3d"

def pct(part: float, whole: float) -> float:
    return float(part / whole * 100) if whole else float("nan")


def read_comments_with_scope(platform: str) -> pd.DataFrame:
    comments = pd.read_csv(PREPROCESSED / f"{platform}_comments.csv", low_memory=False)
    if platform == "reddit":
        comments["scope"] = comments["subreddit"]
    else:
        videos_path = PREPROCESSED / "youtube_videos.csv"
        if videos_path.exists():
            videos = pd.read_csv(videos_path, usecols=lambda c: c in {"video_id", "channel_title"})
            comments = comments.merge(videos.drop_duplicates("video_id"), on="video_id", how="left")
        comments["scope"] = comments.get("channel_title", comments.get("channel_id"))
    return comments


edge_rows = []
threshold_rows = []
thresholds = [1, 5, 10, 30, 100, 250]
community_size_records = []

for platform in ["reddit", "youtube"]:
    summary = json.loads((NETWORKS / f"{platform}_network_summary.json").read_text())
    edge_df = pd.read_csv(NETWORKS / f"{platform}_edges.csv", low_memory=False)
    nodes = pd.read_csv(NETWORKS / f"{platform}_nodes.csv")
    assignments = pd.read_csv(NETWORKS / f"{platform}_community_assignments.csv")
    comments = read_comments_with_scope(platform)
    joined = comments.merge(assignments[["platform", "author_hash", "community"]], on=["platform", "author_hash"], how="left")
    assigned = joined[joined["community"].notna()].copy()
    assigned["community"] = assigned["community"].astype(int)

    collapsed_edges = int(summary["edges"])
    raw_reply_rows = len(edge_df)
    edge_rows.append({
        "platform": platform,
        "raw_reply_edge_rows": raw_reply_rows,
        "collapsed_weighted_author_edges": collapsed_edges,
        "raw_to_collapsed_ratio": raw_reply_rows / collapsed_edges if collapsed_edges else np.nan,
        "nodes": int(summary["nodes"]),
        "largest_wcc_pct": 100 * float(summary["largest_wcc_fraction"]),
        "communities": int(summary["communities"]),
    })

    node_sizes = nodes["community"].value_counts().sort_values(ascending=False)
    comment_sizes = assigned["community"].value_counts().sort_values(ascending=False)
    for community, n in node_sizes.items():
        community_size_records.append({"platform": platform, "unit": "authors", "community": int(community), "size": int(n)})
    for community, n in comment_sizes.items():
        community_size_records.append({"platform": platform, "unit": "comments", "community": int(community), "size": int(n)})

    for threshold in thresholds:
        node_keep = node_sizes[node_sizes >= threshold]
        comment_keep = comment_sizes[comment_sizes >= threshold]
        threshold_rows.append({
            "platform": platform,
            "min_size": threshold,
            "unit": "authors",
            "communities": int(len(node_keep)),
            "pct_communities": pct(len(node_keep), len(node_sizes)),
            "covered_units": int(node_keep.sum()),
            "pct_covered_units": pct(node_keep.sum(), node_sizes.sum()),
        })
        threshold_rows.append({
            "platform": platform,
            "min_size": threshold,
            "unit": "comments",
            "communities": int(len(comment_keep)),
            "pct_communities": pct(len(comment_keep), len(comment_sizes)),
            "covered_units": int(comment_keep.sum()),
            "pct_covered_units": pct(comment_keep.sum(), comment_sizes.sum()),
        })

edge_accounting = pd.DataFrame(edge_rows)
threshold_diagnostics = pd.DataFrame(threshold_rows)
community_size_long = pd.DataFrame(community_size_records)

edge_accounting.to_csv(NETWORKS / "network_edge_accounting.csv", index=False)
threshold_diagnostics.to_csv(NETWORKS / "network_community_thresholds.csv", index=False)
community_size_long.to_csv(NETWORKS / "network_community_size_long.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, unit in zip(axes, ["authors", "comments"]):
    for platform, color in [("reddit", REDDIT_COLOR), ("youtube", YOUTUBE_COLOR)]:
        sizes = community_size_long[(community_size_long["platform"] == platform) & (community_size_long["unit"] == unit)]["size"].sort_values()
        x = np.sort(sizes.to_numpy())
        y = 1 - np.arange(1, len(x) + 1) / len(x)
        ax.step(x, y, where="post", label=platform, color=color, linewidth=2)
    ax.axvline(30, color="grey", linestyle=":", linewidth=1)
    ax.axvline(100, color="grey", linestyle="--", linewidth=1)
    ax.axvline(250, color="black", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(f"Community size measured in {unit}")
    ax.set_title(f"Community-size long tail ({unit})")
    ax.set_ylabel("Share of communities above size")
    ax.legend(title="")
plt.tight_layout()
out = PLOTS / "network_community_size_ccdf.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()

print(f"Wrote {edge_accounting.shape[0]} edge-accounting rows and {threshold_diagnostics.shape[0]} threshold rows.")
edge_accounting

In [ ]:
scope_rows = []
for platform in ["reddit", "youtube"]:
    assignments = pd.read_csv(NETWORKS / f"{platform}_community_assignments.csv")
    comments = read_comments_with_scope(platform)
    author_scope = comments.dropna(subset=["scope"]).groupby("author_hash")["scope"].agg(lambda s: s.mode().iloc[0]).rename("dominant_scope")
    labelled = assignments.merge(author_scope, on="author_hash", how="inner")
    tab = pd.crosstab(labelled["community"], labelled["dominant_scope"])
    scope_rows.append({
        "platform": platform,
        "authors_with_scope": int(len(labelled)),
        "louvain_communities": int(labelled["community"].nunique()),
        "scope_groups": int(labelled["dominant_scope"].nunique()),
        "weighted_scope_purity": float(tab.max(axis=1).sum() / tab.to_numpy().sum()),
        "median_community_scope_share": float((tab.max(axis=1) / tab.sum(axis=1)).median()),
        "pct_communities_at_least_80pct_one_scope": float(((tab.max(axis=1) / tab.sum(axis=1)) >= 0.80).mean() * 100),
    })

scope_baseline = pd.DataFrame(scope_rows)
scope_baseline.to_csv(NETWORKS / "network_scope_baseline_comparison.csv", index=False)

scope_baseline


**Critical reading of the diagnostics.** The network analysis is a structured hand-off to downstream text/community analysis, not a definitive taxonomy of publics. The strongest claims are thresholded and platform-specific: Reddit has enough reciprocal, connected conversation to support community-level interpretation; YouTube is better described as shallow channel/video reply structure with partial community coverage. Louvain is useful because it gives a full author partition, but its added value is reported through notebook 07's coverage and >=100-comment subset.
